In [ ]:
#!pip install lxml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 34.0 MB/s  0:00:00 38.1 MB/s eta 0:00:01

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
import yfinance as yf
import numpy as np
import requests
from pypfopt import EfficientFrontier
from pypfopt import risk_models
from pypfopt import expected_returns

# ------------------------
# 获取S&P500股票列表
# ------------------------

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

headers = {
"User-Agent": "Mozilla/5.0"
}

html = requests.get(url, headers=headers).text

table = pd.read_html(html)[0]

tickers = table["Symbol"].tolist()

print("Total stocks:", len(tickers))

# ------------------------
# 下载价格数据
# ------------------------

price = yf.download(
    tickers,
    start="2018-01-01",
    auto_adjust=True,
    progress=False
)["Close"]

# 删除缺失数据
price = price.dropna(axis=1)

print("Stocks with data:", price.shape[1])

# ------------------------
# 因子计算
# ------------------------

returns = price.pct_change()

momentum = price.pct_change(252).iloc[-1]

volatility = returns.rolling(252).std().iloc[-1]

value = 1 / price.iloc[-1]

def zscore(x):
    return (x - x.mean()) / x.std()

score = (
    0.5*zscore(momentum)
    +0.3*zscore(value)
    -0.2*zscore(volatility)
)

ranking = score.sort_values(ascending=False)

# ------------------------
# 选前15只
# ------------------------

selected = ranking.head(15).index

print("\nSelected stocks:")

print(selected)

# ------------------------
# 组合优化
# ------------------------

selected_prices = price[selected]

mu = expected_returns.mean_historical_return(selected_prices)

S = risk_models.sample_cov(selected_prices)

ef = EfficientFrontier(mu, S)

weights = ef.max_sharpe()

cleaned = ef.clean_weights()

print("\nRecommended portfolio:\n")

for k,v in cleaned.items():
    if v > 0:
        print(k, round(v*100,2), "%")

/var/folders/fk/f1mjs7z94bx25kz_8b8jyp7c0000gn/T/ipykernel_54806/3365441911.py:21: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(html)[0]


Total stocks: 503


$BF.B: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-08)
$BRK.B: possibly delisted; no timezone found

2 Failed downloads:
['BF.B']: possibly delisted; no price data found  (1d 2018-01-01 -> 2026-05-08)
['BRK.B']: possibly delisted; no timezone found


Stocks with data: 474

Selected stocks:
Index(['LITE', 'WDC', 'STX', 'MU', 'CIEN', 'F', 'VTRS', 'INTC', 'HBAN', 'WBD',
       'AES', 'PSKY', 'CAG', 'PCG', 'HST'],
      dtype='object', name='Ticker')

Recommended portfolio:

STX 55.93 %
CIEN 44.07 %


In [15]:
import pandas as pd
import yfinance as yf
import numpy as np
import requests
import matplotlib.pyplot as plt
from pypfopt import EfficientFrontier, risk_models, expected_returns

def get_fundamental_value_metrics(tickers):
    rows = []

    for t in tickers:
        try:
            tk = yf.Ticker(t)
            info = tk.info

            pb = info.get("priceToBook", np.nan)
            pe = info.get("trailingPE", np.nan)

            rows.append({
                "Ticker": t,
                "priceToBook": pb,
                "trailingPE": pe,
            })

        except Exception:
            rows.append({
                "Ticker": t,
                "priceToBook": np.nan,
                "trailingPE": np.nan,
            })

    df = pd.DataFrame(rows).set_index("Ticker")

    # 转成“越大越便宜”的方向
    df["book_to_market"] = 1 / df["priceToBook"]
    df["earnings_yield"] = 1 / df["trailingPE"]

    # 清理异常值
    df = df.replace([np.inf, -np.inf], np.nan)

    return df

# ------------------------
# 获取 S&P500 股票列表
# ------------------------
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {"User-Agent": "Mozilla/5.0"}
html = requests.get(url, headers=headers).text
table = pd.read_html(html)[0]
tickers = table["Symbol"].tolist()

# Yahoo Finance 对部分 ticker 格式要求不同，比如 BRK.B -> BRK-B
tickers = [t.replace(".", "-") for t in tickers]

print("Total stocks:", len(tickers))

# ------------------------
# 下载价格数据
# ------------------------
price = yf.download(
    tickers,
    start="2016-01-01",
    end="2026-03-15",
    auto_adjust=True,
    progress=False
)["Close"]

# 删除缺失过多的股票
price = price.dropna(axis=1, thresh=len(price) * 0.8)
price = price.ffill()

print("Stocks with data:", price.shape[1])

# ------------------------
# 工具函数
# ------------------------
def zscore(x):
    std = x.std()
    if std == 0 or pd.isna(std):
        return pd.Series(0, index=x.index)
    return (x - x.mean()) / std

def compute_scores(hist_price):
    returns = hist_price.pct_change()

    momentum = hist_price.pct_change(252).iloc[-1]
    volatility = returns.rolling(252).std().iloc[-1]
    value = 1 / hist_price.iloc[-1]

    score = (
        0.5 * zscore(momentum)
        + 0.3 * zscore(value)
        - 0.2 * zscore(volatility)
    )

    score = score.replace([np.inf, -np.inf], np.nan).dropna()
    return score.sort_values(ascending=False)

def get_weights(hist_price, top_n=15, weight_cap=0.15):
    ranking = compute_scores(hist_price)
    selected = ranking.head(top_n).index.tolist()

    selected_prices = hist_price[selected].dropna(axis=1)
    selected = selected_prices.columns.tolist()

    if len(selected) < 5:
        return None

    try:
        mu = expected_returns.mean_historical_return(selected_prices)
        S = risk_models.sample_cov(selected_prices)

        ef = EfficientFrontier(mu, S, weight_bounds=(0, weight_cap))
        ef.max_sharpe()
        weights = ef.clean_weights()

        weights = {k: v for k, v in weights.items() if v > 0}
        if len(weights) == 0:
            raise ValueError("No valid optimized weights")

        # 归一化，防止 clean 后总和不等于 1
        total = sum(weights.values())
        weights = {k: v / total for k, v in weights.items()}
        return weights

    except Exception:
        # 如果优化失败，退回等权重
        n = min(top_n, len(selected))
        if n == 0:
            return None
        selected = selected[:n]
        return {k: 1 / n for k in selected}

def portfolio_return(weights, start_prices, end_prices):
    common = [k for k in weights if k in start_prices.index and k in end_prices.index]
    if len(common) == 0:
        return 0

    rets = end_prices[common] / start_prices[common] - 1
    port_ret = sum(weights[k] * rets[k] for k in common)
    return port_ret

# ------------------------
# 生成月度调仓日
# ------------------------
month_ends = price.resample("M").last().index
month_ends = [d for d in month_ends if d in price.index]

# 至少需要 252 个交易日来算 12 个月动量
start_idx = 12
rebalance_dates = month_ends[start_idx:-1]

# ------------------------
# 回测主循环
# ------------------------
results = []

for i, rebalance_date in enumerate(rebalance_dates):
    next_date = month_ends[month_ends.index(rebalance_date) + 1]

    hist_price = price.loc[:rebalance_date].dropna(axis=1, thresh=len(price.loc[:rebalance_date]) * 0.8)
    hist_price = hist_price.ffill()

    if len(hist_price) < 252:
        continue

    weights = get_weights(hist_price, top_n=15, weight_cap=0.15)
    if weights is None:
        continue

    start_prices = price.loc[rebalance_date]
    end_prices = price.loc[next_date]

    ret = portfolio_return(weights, start_prices, end_prices)

    results.append({
        "date": next_date,
        "return": ret,
        "n_stocks": len(weights),
        "weights": weights
    })

    if i % 12 == 0:
        print(f"{rebalance_date.date()} -> {next_date.date()} | return = {ret:.2%}")

# ------------------------
# 结果整理
# ------------------------
bt = pd.DataFrame(results).set_index("date")
bt["equity_curve"] = (1 + bt["return"]).cumprod()

# S&P500 基准（用 SPY 代替）
spy = yf.download("SPY", start="2016-01-01", end="2026-03-15", auto_adjust=True, progress=False)["Close"]
spy_monthly = spy.resample("M").last().pct_change().reindex(bt.index)
benchmark = (1 + spy_monthly.fillna(0)).cumprod()

# ------------------------
# 绩效指标函数
# ------------------------
def annualized_return(r):
    r = pd.Series(r).dropna()
    total_return = (1 + r).prod()
    years = len(r) / 12
    return total_return ** (1 / years) - 1 if years > 0 else np.nan

def annualized_volatility(r):
    r = pd.Series(r).dropna()
    return r.std() * np.sqrt(12)

def sharpe_ratio(r, rf=0.0):
    r = pd.Series(r).dropna()
    vol = annualized_volatility(r)
    if pd.isna(vol) or np.isclose(vol, 0):
        return np.nan
    return (annualized_return(r) - rf) / vol

def max_drawdown(curve):
    curve = pd.Series(curve).dropna()
    rolling_max = curve.cummax()
    drawdown = curve / rolling_max - 1
    return drawdown.min()

# ------------------------
# S&P500 基准（SPY）
# ------------------------
spy = yf.download(
    "SPY",
    start="2016-01-01",
    end="2026-03-15",
    auto_adjust=True,
    progress=False
)["Close"]

if isinstance(spy, pd.DataFrame):
    spy = spy.iloc[:, 0]

spy_monthly = spy.resample("ME").last().pct_change().reindex(bt.index)

if isinstance(spy_monthly, pd.DataFrame):
    spy_monthly = spy_monthly.iloc[:, 0]

bench_ret = spy_monthly.fillna(0)
benchmark = (1 + bench_ret).cumprod()

# ------------------------
# 策略指标
# ------------------------
strategy_ann = annualized_return(bt["return"])
strategy_vol = annualized_volatility(bt["return"])
strategy_sharpe = sharpe_ratio(bt["return"])
strategy_mdd = max_drawdown(bt["equity_curve"])

# ------------------------
# 基准指标
# ------------------------
bench_ann = annualized_return(bench_ret)
bench_vol = annualized_volatility(bench_ret)
bench_sharpe = sharpe_ratio(bench_ret)
bench_mdd = max_drawdown(benchmark)

print("\nStrategy Performance")
print(f"Annualized Return: {strategy_ann:.2%}")
print(f"Annualized Volatility: {strategy_vol:.2%}")
print(f"Sharpe Ratio: {strategy_sharpe:.2f}")
print(f"Max Drawdown: {strategy_mdd:.2%}")

print("\nSPY Benchmark")
print(f"Annualized Return: {bench_ann:.2%}")
print(f"Annualized Volatility: {bench_vol:.2%}")
print(f"Sharpe Ratio: {bench_sharpe:.2f}")
print(f"Max Drawdown: {bench_mdd:.2%}")

/var/folders/fk/f1mjs7z94bx25kz_8b8jyp7c0000gn/T/ipykernel_54764/3629958201.py:14: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(html)[0]


Total stocks: 503
Stocks with data: 478
2017-06-30 -> 2017-07-31 | return = 4.84%


/var/folders/fk/f1mjs7z94bx25kz_8b8jyp7c0000gn/T/ipykernel_54764/3629958201.py:111: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  month_ends = price.resample("M").last().index


2018-11-30 -> 2018-12-31 | return = -6.09%
2020-04-30 -> 2020-06-30 | return = 10.14%
2021-11-30 -> 2021-12-31 | return = 2.87%
2023-02-28 -> 2023-03-31 | return = -0.19%
2024-07-31 -> 2024-09-30 | return = 11.95%

Strategy Performance
Annualized Return: 38.80%
Annualized Volatility: 29.74%
Sharpe Ratio: 1.30
Max Drawdown: -25.73%

SPY Benchmark
Annualized Return: 14.60%
Annualized Volatility: 16.51%
Sharpe Ratio: 0.88
Max Drawdown: -24.44%


/var/folders/fk/f1mjs7z94bx25kz_8b8jyp7c0000gn/T/ipykernel_54764/3629958201.py:159: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  spy_monthly = spy.resample("M").last().pct_change().reindex(bt.index)


In [ ]:
# =============================================================
# 量化投资系统 v2.0 —— 当前选股 + £500 实操买入清单 (Trading 212)
# =============================================================
# 主要改进:
#   1) 价值因子改用真实 P/E、P/B (替代无意义的 1/price)
#   2) 加入质量因子 (ROE > 10%) 与短期反转
#   3) 行业分散约束 (每行业最多 1 只)
#   4) 移除硬编码日期 (使用今日)
#   5) 数据缓存到本地 (避免重复下载)
#   6) GBP -> USD 汇率换算 + T212 0.15% FX 费 + 碎股 (4 位小数) 数量
# =============================================================
import os, pickle
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from io import StringIO
from datetime import datetime

CACHE_DIR = "stock_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

TODAY = pd.Timestamp.today().normalize()
DATA_START = "2018-01-01"
N_STOCKS = 5
GBP_BUDGET = 500.0
T212_FX_FEE = 0.0015  # Trading 212 换汇费 0.15%

# -------------------------------------------------------------
# 1. 获取 S&P 500 名单 + 行业信息
# -------------------------------------------------------------
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
sp_table = pd.read_html(StringIO(html))[0]
sp_table["Symbol"] = sp_table["Symbol"].str.replace(".", "-", regex=False)
tickers = sp_table["Symbol"].tolist()
ticker_sector = dict(zip(sp_table["Symbol"], sp_table["GICS Sector"]))
print(f"S&P 500 股票总数: {len(tickers)}")

# -------------------------------------------------------------
# 2. 价格数据 (带本地缓存, 避免每次重下)
# -------------------------------------------------------------
PRICE_CACHE = os.path.join(CACHE_DIR, "sp500_prices.parquet")

def load_prices():
    if os.path.exists(PRICE_CACHE):
        cached = pd.read_parquet(PRICE_CACHE)
        last_date = cached.index[-1].normalize()
        if (TODAY - last_date).days <= 2:
            print(f"使用缓存价格 (最新: {last_date.date()})")
            return cached
    print("下载价格中...")
    df = yf.download(tickers, start=DATA_START, end=TODAY,
                     auto_adjust=True, progress=False)["Close"]
    df.to_parquet(PRICE_CACHE)
    return df

price = load_prices()
price = price.dropna(axis=1, thresh=int(len(price) * 0.9))  # 至少90%数据
price = price.ffill()
print(f"清洗后股票数: {price.shape[1]}")

# -------------------------------------------------------------
# 3. 价格因子打分: 动量(12-1) + 低波 + 短期反转
# -------------------------------------------------------------
returns = price.pct_change()
mom_12_1   = price.shift(21).pct_change(231).iloc[-1]   # 跳过最近 1 月, 取 12-1 动量
vol_1y     = returns.rolling(252).std().iloc[-1]
rev_1m     = -returns.iloc[-21:].sum()                  # 短期反转

def zscore(s):
    s = s.dropna()
    return (s - s.mean()) / s.std(ddof=0)

price_score = (
    0.6 * zscore(mom_12_1)
    - 0.3 * zscore(vol_1y)
    + 0.1 * zscore(rev_1m)
).dropna().sort_values(ascending=False)

candidates = price_score.head(50).index.tolist()
print(f"\n价格因子前 50 候选池已生成")

# -------------------------------------------------------------
# 4. 基本面数据 (PE / PB / ROE / 行业) —— 仅用于当前选股, 不参与回测
# -------------------------------------------------------------
FUND_CACHE = os.path.join(CACHE_DIR, "fundamentals.pkl")

def get_fundamentals(tk_list, max_age_days=3):
    if os.path.exists(FUND_CACHE):
        with open(FUND_CACHE, "rb") as f:
            cached = pickle.load(f)
        if (TODAY - cached["asof"]).days <= max_age_days:
            print("使用缓存基本面数据")
            return cached["data"]

    print(f"下载基本面数据 ({len(tk_list)} 只, 约 30-60 秒)...")
    rows = []
    for i, t in enumerate(tk_list):
        try:
            info = yf.Ticker(t).info
            rows.append({
                "Ticker": t,
                "PE": info.get("trailingPE", np.nan),
                "PB": info.get("priceToBook", np.nan),
                "ROE": info.get("returnOnEquity", np.nan),
                "MarketCap": info.get("marketCap", np.nan),
                "Sector": info.get("sector", ticker_sector.get(t, "Unknown")),
            })
        except Exception:
            rows.append({"Ticker": t, "PE": np.nan, "PB": np.nan,
                         "ROE": np.nan, "MarketCap": np.nan,
                         "Sector": ticker_sector.get(t, "Unknown")})
        if (i + 1) % 10 == 0:
            print(f"  {i+1}/{len(tk_list)}", end="\r")
    df = pd.DataFrame(rows).set_index("Ticker")
    with open(FUND_CACHE, "wb") as f:
        pickle.dump({"asof": TODAY, "data": df}, f)
    return df

fund = get_fundamentals(candidates)

# -------------------------------------------------------------
# 5. 质量过滤 + 综合打分 + 行业分散
# -------------------------------------------------------------
quality = (
    (fund["PE"] > 0) & (fund["PE"] < 50) &   # 盈利且不过热
    (fund["ROE"] > 0.10) &                    # ROE > 10%
    (fund["PB"] > 0)
)
qualified = fund[quality].index.tolist()
print(f"\n质量过滤后剩余: {len(qualified)} 只")

if len(qualified) < N_STOCKS:
    print("⚠️  合格股票数过少, 放宽过滤标准")
    quality = (fund["PE"] > 0) & (fund["ROE"] > 0.05) & (fund["PB"] > 0)
    qualified = fund[quality].index.tolist()

ep = (1 / fund.loc[qualified, "PE"])
bp = (1 / fund.loc[qualified, "PB"])

final_score = (
    0.5 * zscore(price_score.loc[qualified])
    + 0.3 * zscore(ep)
    + 0.2 * zscore(bp)
).sort_values(ascending=False)

selected, sectors_used = [], set()
for t in final_score.index:
    sec = fund.loc[t, "Sector"]
    if sec not in sectors_used:
        selected.append(t)
        sectors_used.add(sec)
    if len(selected) == N_STOCKS:
        break

print(f"\n{'='*72}")
print(f"{'最终选股 (行业分散, 每行业最多 1 只)':^72}")
print(f"{'='*72}")
print(f"{'Ticker':<8}{'Sector':<26}{'PE':>8}{'PB':>8}{'ROE':>10}{'Score':>10}")
print("-"*72)
for t in selected:
    print(f"{t:<8}{fund.loc[t,'Sector']:<26}"
          f"{fund.loc[t,'PE']:>8.2f}{fund.loc[t,'PB']:>8.2f}"
          f"{fund.loc[t,'ROE']:>10.2%}{final_score[t]:>10.2f}")

# -------------------------------------------------------------
# 6. £500 实操买入清单 (Trading 212, 等权 + 碎股)
# -------------------------------------------------------------
fx_data = yf.Ticker("GBPUSD=X").history(period="5d")["Close"]
fx_rate = float(fx_data.iloc[-1])

gbp_after_fx = GBP_BUDGET * (1 - T212_FX_FEE)
usd_total    = gbp_after_fx * fx_rate
gbp_per      = gbp_after_fx / N_STOCKS
usd_per      = usd_total / N_STOCKS

current_prices = price[selected].iloc[-1]

print(f"\n{'='*72}")
print(f"{'£500 BUY LIST (Trading 212)':^72}")
print(f"{'='*72}")
print(f"GBP/USD 汇率: {fx_rate:.4f}    FX 费 (0.15%): £{GBP_BUDGET-gbp_after_fx:.2f}")
print(f"可用资金:     £{gbp_after_fx:.2f}  =  ${usd_total:.2f}")
print(f"每只股票:     £{gbp_per:.2f}     =  ${usd_per:.2f}\n")
print(f"{'Ticker':<8}{'Price ($)':>12}{'Shares':>12}{'Cost ($)':>12}{'Cost (£)':>12}")
print("-"*72)

total_usd = 0.0
buy_orders = []
for t in selected:
    p = float(current_prices[t])
    shares = round(usd_per / p, 4)             # T212 支持 4 位碎股
    cost_usd = shares * p
    cost_gbp = cost_usd / fx_rate
    total_usd += cost_usd
    buy_orders.append({"ticker": t, "price_usd": p, "shares": shares,
                       "cost_usd": cost_usd, "cost_gbp": cost_gbp})
    print(f"{t:<8}{p:>12.2f}{shares:>12.4f}{cost_usd:>12.2f}{cost_gbp:>12.2f}")

total_gbp = total_usd / fx_rate
print("-"*72)
print(f"{'TOTAL':<8}{'':>12}{'':>12}{total_usd:>12.2f}{total_gbp:>12.2f}")
print(f"{'FX fee':<8}{'':>12}{'':>12}{'':>12}{GBP_BUDGET-gbp_after_fx:>12.2f}")
print(f"{'剩余':<8}{'':>12}{'':>12}{'':>12}{GBP_BUDGET-total_gbp-(GBP_BUDGET-gbp_after_fx):>12.2f}")
print("="*72)
print("\n操作步骤 (Trading 212):")
print("  1. 在 App 内将 £500 转入 Stocks ISA / Invest 账户")
print("  2. 搜索每个 ticker, 选择 'Buy by amount', 输入对应的 £ 金额或 Shares")
print("  3. 建议设置月度自动调仓提醒")


In [ ]:
# =============================================================
# v2.0 修正后的回测
# =============================================================
# 注意:
#   - 基本面因子(PE/PB/ROE)无历史数据, 回测仅用价格因子(动量+低波+反转)
#   - 加入行业分散约束, 5 只等权
#   - 加入交易成本 (T212 几乎 0 手续费, 但保留 0.05% 滑点估计)
#   - Sharpe 用真实无风险利率 4.3% (10Y 国债)
#   - 仍存在幸存者偏差 (用当前 S&P 500 名单), 实盘真实收益会更低
# =============================================================
import matplotlib.pyplot as plt

TRANSACTION_COST = 0.0005  # 单边 0.05% (T212 滑点估计, 0 手续费)
RF_RATE = 0.043            # 4.3% 无风险利率

def factor_score_at(hist):
    rets = hist.pct_change()
    mom  = hist.shift(21).pct_change(231).iloc[-1]
    vol  = rets.rolling(252).std().iloc[-1]
    rev  = -rets.iloc[-21:].sum()
    s = (0.6 * zscore(mom) - 0.3 * zscore(vol) + 0.1 * zscore(rev))
    return s.dropna().sort_values(ascending=False)

def select_diversified(score, sector_map, n=5):
    sel, used = [], set()
    for t in score.index:
        sec = sector_map.get(t)
        if sec is None or sec in used:
            continue
        sel.append(t); used.add(sec)
        if len(sel) == n: break
    return sel

month_ends = price.resample("ME").last().index
month_ends = [d for d in month_ends if d in price.index]
rebalance_dates = month_ends[12:-1]

results, prev_w = [], {}
for rd in rebalance_dates:
    next_d = month_ends[month_ends.index(rd) + 1]
    hist = price.loc[:rd].dropna(axis=1, thresh=int(len(price.loc[:rd]) * 0.9)).ffill()
    if len(hist) < 252:
        continue
    score = factor_score_at(hist)
    sel = select_diversified(score, ticker_sector, n=N_STOCKS)
    if len(sel) < N_STOCKS:
        continue

    w = {t: 1/N_STOCKS for t in sel}
    sp_, ep_ = price.loc[rd, sel], price.loc[next_d, sel]
    gross = sum(w[t] * (ep_[t]/sp_[t] - 1) for t in sel
                if pd.notna(sp_[t]) and pd.notna(ep_[t]))

    keys = set(w) | set(prev_w)
    turnover = sum(abs(w.get(t, 0) - prev_w.get(t, 0)) for t in keys) / 2
    cost = turnover * TRANSACTION_COST * 2
    net = gross - cost

    results.append({"date": next_d, "gross": gross, "net": net, "turnover": turnover})
    prev_w = w

bt = pd.DataFrame(results).set_index("date")
bt["eq_gross"] = (1 + bt["gross"]).cumprod()
bt["eq_net"]   = (1 + bt["net"]).cumprod()

# SPY 基准
spy = yf.download("SPY", start=DATA_START, end=TODAY,
                  auto_adjust=True, progress=False)["Close"]
if isinstance(spy, pd.DataFrame):
    spy = spy.iloc[:, 0]
spy_m = spy.resample("ME").last().pct_change().reindex(bt.index).fillna(0)
bt["eq_spy"] = (1 + spy_m).cumprod()

def metrics(rets):
    rets = pd.Series(rets).dropna()
    ann   = (1+rets).prod()**(12/len(rets)) - 1
    vol   = rets.std() * np.sqrt(12)
    shp   = (ann - RF_RATE) / vol if vol > 0 else np.nan
    eq    = (1+rets).cumprod()
    mdd   = (eq / eq.cummax() - 1).min()
    return ann, vol, shp, mdd

m_g = metrics(bt["gross"])
m_n = metrics(bt["net"])
m_s = metrics(spy_m)

print(f"\n{'='*78}")
print(f"{'策略':<26}{'年化收益':>12}{'年化波动':>12}{'Sharpe':>10}{'最大回撤':>14}")
print("-"*78)
for name, m in [("v2.0 (毛收益)", m_g),
                ("v2.0 (扣 0.05% 成本)", m_n),
                ("SPY 基准", m_s)]:
    print(f"{name:<26}{m[0]:>11.2%}{m[1]:>11.2%}{m[2]:>9.2f}{m[3]:>13.2%}")
print("="*78)
print(f"\n平均月度换手率: {bt['turnover'].mean():.2%}")
print("\n⚠️  幸存者偏差: 回测使用当前 S&P 500 名单, 实盘真实收益会更低 (估计 -3% 到 -5% 年化)")
print("⚠️  基本面因子未参与回测; 当前选股 (上一个 cell) 已使用质量+价值过滤")

plt.figure(figsize=(12, 5))
plt.plot(bt.index, bt["eq_gross"], label="v2.0 (gross)", lw=1.6)
plt.plot(bt.index, bt["eq_net"],   label="v2.0 (net of costs)", lw=1.6)
plt.plot(bt.index, bt["eq_spy"],   label="SPY", lw=1.6, ls="--", color="gray")
plt.title("v2.0 (5 stocks, sector-diversified) vs SPY")
plt.ylabel("Equity (1 = start)")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.show()
